# Dependencies

Install the required dependencies and libraries

In [ ]:
%pip install langchain langchain_core langchain-huggingface ipywidgets python-dotenv

# Environment Variables and Constants

Initialize the constants and environment variables first

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")

HFH_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

SYSTEM_MSG = """
You are an AI security analyst.
You will be provided vulnerability scanner data in SARIF format with vulnerabilites detected in assets and services.
Alongside that, you will also be provided with the artifacts information such as assets information and service and architecture information and description.
Your job is to analyze the actual risk posed by the vulnerabilites by taking in the contextual information from the artifacts.
"""

HUMAN_MSG = """
Scanner: {scanner}

Assets: {assets}

Exploit Data: {exploit}

SBOM: {sbom}

Architecture: {architecture}

Service Documentation: {service_docs}

Service Catalogue: {service_catalogue}
"""

# Requirements

Install the requirements and import relevant modules

In [ ]:
import json
from pathlib import Path
from typing import Dict, List

from langchain.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# Data Loaders

Methods used for loading the data

In [ ]:
def load_json_files() -> Dict:
    """
    Loads vulnerability and asset data from the json files

    Returns:
        Dictionary of the data files with their data content
    """

    # A dictionary to hold different datasets
    data_registry = {}
    
    # Path to data folder
    data_folder = Path("data")
    
    for file_path in data_folder.glob("*.json"):
        with open(file_path, 'r', encoding='utf-8') as f:

            data_registry[file_path.stem] = json.load(f)

    return data_registry

# Workflow

These methods setup the basic model workflow

In [ ]:
def create_prompt(sys_msg: str, hum_msg: str) -> ChatPromptTemplate:
    """
    Sets up the chat prompt to be used with the model

    Args:
        * sysm_msg (str): The system message template
        * hum_msg (str): The human message prompt

    Returns:
        `ChatPromptTemplate` object
    """

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", sys_msg),
        ("human", hum_msg)
    ])
    return chat_prompt


def load_model_from_hf(repo_id: str) -> ChatHuggingFace:
    """
    Connects to the model using the HuggingFace Inference API.

    Returns:
        A `ChatHuggingFace` model
    """

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )
    model = ChatHuggingFace(llm=llm)
    return model

In [ ]:
model = load_model_from_hf("deepseek-ai/DeepSeek-R1")

In [ ]:
prompt = create_prompt("you are a comedian", "Tell me a joke about {joke}")